[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/reza0mowlavi/huggingface-llm-basics/blob/main/notebooks/1.Basics%20of%20HuggingFace.ipynb)


## Colab Setup Note

Some of the notebooks require extra Python libraries that may not be installed in a fresh Colab runtime, or may need to be upgraded to a compatible version.

If you are running this notebook in Colab, please install the required dependencies in the setup cell at the before continuing.

In [ ]:
!pip install --upgrade datasets transformers

# LLM Inference Workshop – Part 1
## HuggingFace Basics and Model Internals

In this notebook we will:

1. Understand how `.from_pretrained()` works.
2. See the difference between `AutoModel*`, `BertFor*`, and `*ForCausalLM` classes.
3. Build a tiny custom model using `PretrainedConfig` and `PreTrainedModel`.
4. Save and load a model with `save_pretrained()` and `from_pretrained()`.

We keep models **small** to make the notebook fast on CPU/GPU.


## What Is HuggingFace and Why It Matters

HuggingFace is an open ecosystem for working with modern NLP and LLMs. In practice it gives you:

- **Model hub + checkpoints**: a centralized place to share and download pretrained models.
- **Standardized APIs**: `from_pretrained`, `save_pretrained`, tokenizers, and configs that work across architectures.
- **Training and inference tooling**: `Trainer`, PEFT adapters, evaluation helpers, and deployment‑ready runtimes.
- **Community reuse**: the same code can load tiny CPU models or large GPU models with minimal changes.

Why this matters in a course setting:

- You can focus on **concepts** (tokenization, generation, fine‑tuning) without reinventing infrastructure.
- Experiments are **reproducible** because config + weights + tokenizer are versioned and portable.
- You can scale from **toy examples to real LLMs** with the same mental model and APIs.

In [2]:
from typing import Literal
from dataclasses import dataclass
from pathlib import Path

import torch
from torch import nn

from datasets import load_dataset


from transformers import (
    AutoConfig,
    AutoModel,
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    BertForSequenceClassification,
    PretrainedConfig,
    PreTrainedModel,
    Trainer,
    TrainingArguments,
)
from transformers.utils import ModelOutput


In [ ]:
torch.manual_seed(0)

if torch.cuda.is_available():
    device = "cuda"
    dtype = "auto" ## Use torch.float32 in Colab
elif torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float32
else:
    device = "cpu"
    dtype = torch.float32

print("Device:", device)
print("Dtype:", dtype)


Device: cuda
Dtype: auto


## What `from_pretrained()` Does

`from_pretrained()` does three main things:

1. **Downloads (or loads from cache) the config** for a checkpoint.
2. **Constructs the correct model class** based on that config.
3. **Loads weights** into the model.

The cache location is typically under `~/.cache/huggingface` (unless you set
`HF_HOME` or `TRANSFORMERS_CACHE`).


In [4]:
MODEL_NAME = "sshleifer/tiny-gpt2"

# Load only the config
cfg = AutoConfig.from_pretrained(MODEL_NAME)
print("model_type:", cfg.model_type)
print("architectures:", cfg.architectures)

# Load full model
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to(device)
model.eval()

print("Loaded class:", model.__class__.__name__)
print("Num parameters:", sum(p.numel() for p in model.parameters()))


model_type: gpt2
architectures: ['GPT2LMHeadModel']
Loaded class: GPT2LMHeadModel
Num parameters: 102714


In [5]:
# Save locally and load back
save_dir = Path("artifacts/tiny_gpt2")
save_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained(save_dir)

reloaded = AutoModelForCausalLM.from_pretrained(save_dir, dtype=dtype).to(device)
reloaded.eval()

print("Reloaded class:", reloaded.__class__.__name__)


/nfs/home/seymol/conda_envs/v3.13/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/nfs/home/seymol/conda_envs/v3.13/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


Reloaded class: GPT2LMHeadModel


## Model Families and Task Heads

Transformer architectures split into three common families:

- **Encoder-only** (BERT-style): classification, token labeling, QA, masked LM.
- **Decoder-only** (GPT/LLM-style): causal LM (next-token prediction).
- **Encoder-decoder** (T5/BART-style): seq2seq generation.

Task-specific heads are exposed via `*For*` classes:

- Encoder-only examples: `BertForSequenceClassification`, `BertForTokenClassification`, `BertForMaskedLM`.
- Decoder-only examples: `GPT2LMHeadModel`, `LlamaForCausalLM`, `AutoModelForCausalLM`.
- Encoder-decoder examples: `AutoModelForSeq2SeqLM`.

The `AutoModel*` classes select the correct concrete class based on the checkpoint config.


In [6]:
import transformers
transformers.*For*?

🚨 `high_res_pixel_values` is part of DeepseekVLHybridForConditionalGeneration.forward's signature, but not documented. Make sure to add it to the docstring of the function in /nfs/home/seymol/conda_envs/v3.13/lib/python3.13/site-packages/transformers/models/deepseek_vl_hybrid/modeling_deepseek_vl_hybrid.py.
🚨 Config not found for parakeet. You can manually add it to HARDCODED_CONFIG_FOR_MODELS in utils/auto_docstring.py
🚨 Config not found for parakeet. You can manually add it to HARDCODED_CONFIG_FOR_MODELS in utils/auto_docstring.py
🚨 Config not found for parakeet. You can manually add it to HARDCODED_CONFIG_FOR_MODELS in utils/auto_docstring.py


transformers.ASTForAudioClassification
transformers.AlbertForMaskedLM
transformers.AlbertForMultipleChoice
transformers.AlbertForPreTraining
transformers.AlbertForQuestionAnswering
transformers.AlbertForSequenceClassification
transformers.AlbertForTokenClassification
transformers.ApertusForCausalLM
transformers.ApertusForTokenClassification
transformers.ArceeForCausalLM
transformers.ArceeForQuestionAnswering
transformers.ArceeForSequenceClassification
transformers.ArceeForTokenClassification
transformers.AriaForConditionalGeneration
transformers.AriaTextForCausalLM
transformers.AutoModelForAudioClassification
transformers.AutoModelForAudioFrameClassification
transformers.AutoModelForAudioTokenization
transformers.AutoModelForAudioXVector
transformers.AutoModelForCTC
transformers.AutoModelForCausalLM
transformers.AutoModelForDepthEstimation
transformers.AutoModelForDocumentQuestionAnswering
transformers.AutoModelForImageClassification
transformers.AutoModelForImageSegmentation
transform

In [7]:
BERT_TINY = "google/bert_uncased_L-2_H-128_A-2"

# Generic encoder without a task head
encoder = AutoModel.from_pretrained(BERT_TINY)

# Encoder with a classification head
clf = AutoModelForSequenceClassification.from_pretrained(BERT_TINY)

# Encoder with a masked LM head
mlm = AutoModelForMaskedLM.from_pretrained(BERT_TINY)

print("AutoModel class:", encoder.__class__.__name__)
print("SequenceClassification head:", clf.classifier.__class__.__name__)
print("MaskedLM head:", mlm.cls.__class__.__name__)


loading configuration file config.json from cache at /nfs/home/seymol/cache/hf_hub_cache/models--google--bert_uncased_L-2_H-128_A-2/snapshots/30b0a37ccaaa32f332884b96992754e246e48c5f/config.json
Model config BertConfig {
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 128,
  "initializer_range": 0.02,
  "intermediate_size": 512,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 2,
  "num_hidden_layers": 2,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.57.6",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading weights file model.safetensors from cache at /nfs/home/seymol/cache/hf_hub_cache/models--google--bert_uncased_L-2_H-128_A-2/snapshots/30b0a37ccaaa32f332884b96992754e246e48c5f/model.safetensors
Some weights of the model checkpoint at google/bert_uncased_L-2_H-128

AutoModel class: BertModel
SequenceClassification head: Linear
MaskedLM head: BertOnlyMLMHead


In [8]:
# The explicit class is equivalent to the AutoModel mapping
bert_sc = BertForSequenceClassification.from_pretrained(BERT_TINY)
print("Explicit class:", bert_sc.__class__.__name__)


loading configuration file config.json from cache at /nfs/home/seymol/cache/hf_hub_cache/models--google--bert_uncased_L-2_H-128_A-2/snapshots/30b0a37ccaaa32f332884b96992754e246e48c5f/config.json
Model config BertConfig {
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 128,
  "initializer_range": 0.02,
  "intermediate_size": 512,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 2,
  "num_hidden_layers": 2,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.57.6",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading weights file model.safetensors from cache at /nfs/home/seymol/cache/hf_hub_cache/models--google--bert_uncased_L-2_H-128_A-2/snapshots/30b0a37ccaaa32f332884b96992754e246e48c5f/model.safetensors
Some weights of the model checkpoint at google/bert_uncased_L-2_H-128

Explicit class: BertForSequenceClassification


## Implementing a Custom HF Model

Below is a tiny MLP model wrapped in `PreTrainedModel`.
We implement:

- A `PretrainedConfig` class
- A backbone module
- A `ModelOutput`
- The model wrapper with `.forward()` and loss

This is the minimal pattern HuggingFace expects for save/load to work.


In [9]:
TASK = Literal["regression", "classification"]

ACT = {
    "relu": nn.functional.relu,
    "gelu": nn.functional.gelu,
    "sigmoid": nn.functional.sigmoid,
}


class MLPConfig(PretrainedConfig):
    model_type = "mlp"

    def __init__(
        self,
        input_size: int = 8,
        output_size: int = 2,
        hidden_sizes: list[int] | None = None,
        task: TASK = "regression",
        activation: str = "relu",
        **kwargs,
    ):
        super().__init__(**kwargs)

        if task not in {"regression", "classification"}:
            raise ValueError("task must be 'regression' or 'classification'")
        if activation not in ACT:
            raise ValueError(f"Unknown activation: {activation}")

        self.input_size = input_size
        self.output_size = output_size

        self.hidden_sizes = hidden_sizes or []
        self.activation = activation
        self.task = task


class MLP(nn.Module):
    def __init__(self, config: MLPConfig):
        super().__init__()

        self.config = config
        self.hidden_act = ACT[config.activation]

        sizes = [config.input_size, *config.hidden_sizes, config.output_size]
        self.layers = nn.ModuleList(
            nn.Linear(in_f, out_f) for in_f, out_f in zip(sizes[:-1], sizes[1:])
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers[:-1]:
            x = self.hidden_act(layer(x))
        return self.layers[-1](x)


@dataclass
class MLPOutput(ModelOutput):
    logits: torch.Tensor = None
    loss: torch.Tensor | None = None


class MLPModel(PreTrainedModel):
    config_class = MLPConfig

    def __init__(self, config: MLPConfig):
        super().__init__(config)
        self.mlp = MLP(config)
        self.post_init()  # HF weight init

    def _compute_loss(self, logits: torch.Tensor, target: torch.Tensor):
        if self.config.task == "regression":
            return nn.functional.mse_loss(logits, target)
        if self.config.output_size == 1:
            return nn.functional.binary_cross_entropy_with_logits(logits, target)
        return nn.functional.cross_entropy(logits, target)

    def forward(self, x: torch.Tensor, labels: torch.Tensor | None = None) -> MLPOutput:
        logits = self.mlp(x)
        loss = None
        if labels is not None:
            loss = self._compute_loss(logits, labels)
        return MLPOutput(loss=loss, logits=logits)


### Initialize the MLP Classifier

We set `input_size` to match the hashed feature vector.
We choose a small two-hidden-layer MLP for speed.


In [10]:
# Instantiate and run a forward pass
config = MLPConfig(
    input_size=8,
    output_size=2,
    hidden_sizes=[16, 16],
    task="classification",
    activation="gelu",
)
model = MLPModel(config)

x = torch.randn(4, 8)
labels = torch.tensor([0, 1, 0, 1])

out = model(x=x, labels=labels)
print("logits shape:", out.logits.shape)
print("loss:", out.loss.item())


logits shape: torch.Size([4, 2])
loss: 0.6931537389755249


## Saving and Loading Custom Models

Because `MLPModel` is a `PreTrainedModel`, we can use the same HF APIs.


In [11]:
save_dir = Path("artifacts/mlp_demo")
save_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained(save_dir)

loaded = MLPModel.from_pretrained(save_dir)
print("Loaded OK:", isinstance(loaded, MLPModel))

# Check weights are present
print("State dict keys:", list(loaded.state_dict().keys())[:5])


Configuration saved in artifacts/mlp_demo/config.json
Model weights saved in artifacts/mlp_demo/model.safetensors
loading configuration file artifacts/mlp_demo/config.json
Model config MLPConfig {
  "activation": "gelu",
  "architectures": [
    "MLPModel"
  ],
  "dtype": "float32",
  "hidden_sizes": [
    16,
    16
  ],
  "input_size": 8,
  "model_type": "mlp",
  "output_size": 2,
  "task": "classification",
  "transformers_version": "4.57.6"
}

loading weights file artifacts/mlp_demo/model.safetensors


Loaded OK: True
State dict keys: ['mlp.layers.0.weight', 'mlp.layers.0.bias', 'mlp.layers.1.weight', 'mlp.layers.1.bias', 'mlp.layers.2.weight']


## (Optional) Register for AutoModel

If you want `AutoModel.from_pretrained()` to work with your custom model,
you can register it at runtime. This is useful in research code.


In [12]:
from transformers import AutoModel, AutoConfig

AutoConfig.register("mlp", MLPConfig)
AutoModel.register(MLPConfig, MLPModel)

auto_loaded = AutoModel.from_pretrained(save_dir)
print("AutoModel loaded:", auto_loaded.__class__.__name__)


loading configuration file artifacts/mlp_demo/config.json
Model config MLPConfig {
  "activation": "gelu",
  "architectures": [
    "MLPModel"
  ],
  "dtype": "float32",
  "hidden_sizes": [
    16,
    16
  ],
  "input_size": 8,
  "model_type": "mlp",
  "output_size": 2,
  "task": "classification",
  "transformers_version": "4.57.6"
}

loading weights file artifacts/mlp_demo/model.safetensors


AutoModel loaded: MLPModel


## Trainer: Quick Introduction

The `Trainer` API provides a high-level training loop with:

- Automatic batching and gradient steps
- Evaluation and logging
- Checkpointing

We will train our custom MLP on a **tiny** subset of IMDB sentiment.
This is just a toy example to show how to plug a custom model into `Trainer`.


### Tokenizer and Dataset (Quick Preview)

We briefly introduce them here to make the training flow clear.
We will cover them **in depth** in later notebooks.

- **Dataset**: A collection of labeled examples. We use IMDB sentiment here.
- **Tokenizer**: Converts raw text into integer IDs. We only use it here
  to build simple features; later we use tokenization directly for models.

We use the **Hashing Trick** to map token IDs into a fixed-size vector.

To keep this fast, we only take a tiny subset.


In [13]:
# Load IMDB and take a tiny subset
raw = load_dataset("imdb")
train_raw = raw["train"].shuffle(seed=0)
test_raw = raw["test"].shuffle(seed=0)

# A tiny tokenizer just for feature hashing
tok = AutoTokenizer.from_pretrained("google/bert_uncased_L-2_H-128_A-2")

INPUT_SIZE = 1024

def featurize(example):
    # Convert text to token IDs
    ids = tok(example["text"], truncation=True, max_length=1024)["input_ids"]

    # Simple hashed bag-of-words into a fixed vector
    vec = [0.0] * INPUT_SIZE
    for i in ids:
        vec[i % INPUT_SIZE] += 1.0

    # Normalize to reduce length bias
    norm = sum(vec)
    if norm > 0:
        vec = [v / norm for v in vec]

    return {"x": vec, "labels": example["label"]}

train_ds = train_raw.map(featurize, remove_columns=["text"])
test_ds = test_raw.map(featurize, remove_columns=["text"])

train_ds.set_format(type="torch", columns=["x", "labels"])
test_ds.set_format(type="torch", columns=["x", "labels"])


Could not locate the tokenizer configuration file, will try to use the model config instead.
loading configuration file config.json from cache at /nfs/home/seymol/cache/hf_hub_cache/models--google--bert_uncased_L-2_H-128_A-2/snapshots/30b0a37ccaaa32f332884b96992754e246e48c5f/config.json
Model config BertConfig {
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 128,
  "initializer_range": 0.02,
  "intermediate_size": 512,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 2,
  "num_hidden_layers": 2,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.57.6",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}



vocab.txt: 0.00B [00:00, ?B/s]

loading file vocab.txt from cache at /nfs/home/seymol/cache/hf_hub_cache/models--google--bert_uncased_L-2_H-128_A-2/snapshots/30b0a37ccaaa32f332884b96992754e246e48c5f/vocab.txt
loading file tokenizer.json from cache at None
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at None
loading file chat_template.jinja from cache at None
loading configuration file config.json from cache at /nfs/home/seymol/cache/hf_hub_cache/models--google--bert_uncased_L-2_H-128_A-2/snapshots/30b0a37ccaaa32f332884b96992754e246e48c5f/config.json
Model config BertConfig {
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 128,
  "initializer_range": 0.02,
  "intermediate_size": 512,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 2,
  "num_hidden_layers": 2,
  

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [14]:
# Quick label balance check
train_labels = train_raw["label"]
test_labels = test_raw["label"]
print('Train label mean:', sum(train_labels)/len(train_labels))
print('Test label mean:', sum(test_labels)/len(test_labels))


Train label mean: 0.5
Test label mean: 0.5


In [15]:
# Initialize our custom model
config = MLPConfig(
    input_size=INPUT_SIZE,
    output_size=1,
    hidden_sizes=[128, 64],
    task="classification",
    activation="gelu",
)
model = MLPModel(config)


### Trainer and Custom Loss (Optional)

By default, `Trainer` calls your model and looks for `outputs.loss`.
Because our `MLPModel` computes a loss when `labels` are provided,
we **do not need** to override `compute_loss()` for this example.

We show a custom `Trainer` subclass anyway to demonstrate how you
would override the loss when needed (e.g., label smoothing,
multi-task loss, etc.).


In [ ]:
# Custom Trainer to override loss computation
class MLPTrainer(Trainer):
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs: bool = False,
        num_items_in_batch: torch.Tensor | None = None,
    ):
        labels = inputs.pop("labels").float()
        outputs = model(**inputs)
        logits = outputs.logits.view(-1)
        loss = nn.functional.binary_cross_entropy_with_logits(logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = logits.reshape(-1)
    labels = labels.reshape(-1)
    preds = (logits > 0.0).astype(int)
    acc = (preds == labels).mean().item()
    return {"accuracy": acc}


args = TrainingArguments(
    output_dir="artifacts/mlp_imdb",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=1e-3,
    warmup_ratio=0.10,
)

trainer = MLPTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).


In [17]:
train_ds["labels"]

Column([tensor(1), tensor(1), tensor(0), tensor(1), tensor(0), ...])

### Train

This is a tiny run just to demonstrate the API. Accuracy is not the goal.


In [18]:
trainer.train()

***** Running training *****
  Num examples = 25,000
  Num Epochs = 10
  Instantaneous batch size per device = 32
  Total train batch size (w. parallel, distributed & accumulation) = 32
  Gradient Accumulation steps = 1
  Total optimization steps = 7,820
  Number of trainable parameters = 139,521
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Epoch,Training Loss,Validation Loss,Accuracy
1,0.424900,0.475342,0.769960
2,0.387300,0.405527,0.819000
3,0.381200,0.404836,0.821160
4,0.391900,0.412164,0.814800
5,0.368800,0.416518,0.815720
6,0.363600,0.409160,0.819600
7,0.361300,0.413015,0.817360
8,0.363600,0.411255,0.818840
9,0.357400,0.411778,0.818080
10,0.360900,0.412713,0.817960



***** Running Evaluation *****
  Num examples = 25000
  Batch size = 32

***** Running Evaluation *****
  Num examples = 25000
  Batch size = 32

***** Running Evaluation *****
  Num examples = 25000
  Batch size = 32

***** Running Evaluation *****
  Num examples = 25000
  Batch size = 32

***** Running Evaluation *****
  Num examples = 25000
  Batch size = 32

***** Running Evaluation *****
  Num examples = 25000
  Batch size = 32

***** Running Evaluation *****
  Num examples = 25000
  Batch size = 32

***** Running Evaluation *****
  Num examples = 25000
  Batch size = 32

***** Running Evaluation *****
  Num examples = 25000
  Batch size = 32

***** Running Evaluation *****
  Num examples = 25000
  Batch size = 32


Training completed. Do not forget to share your model on huggingface.co/models =)




TrainOutput(global_step=7820, training_loss=0.3922171237218715, metrics={'train_runtime': 38.4507, 'train_samples_per_second': 6501.829, 'train_steps_per_second': 203.377, 'total_flos': 0.0, 'train_loss': 0.3922171237218715, 'epoch': 10.0})

### Evaluate

In [19]:
@torch.no_grad()
def pred_accuracy(model, dataset):
    model.eval()
    x = dataset["x"][:].to(device=model.device)
    logits = model(x)["logits"]
    preds = (logits > 0).int().cpu().flatten()
    labels = dataset["labels"][:].int().cpu().flatten()
    return preds, (preds == labels).float().mean().item()

In [20]:
preds, acc = pred_accuracy(model, test_ds)
print(acc)

0.8179600238800049


## Summary

You now saw:

- What `.from_pretrained()` does under the hood.
- The difference between base models and task-specific heads.
- A minimal custom model that integrates with the HF ecosystem.
- How to save and load both official and custom models.
- How to train and evaluate a custom model using `Trainer`.
